# Employee Attrition: training pipeline

เทรนโมเดลจาก 24 ฟิลด์เดียวกับที่ `POST /predict` รับ แล้วบันทึกทุกอย่างที่ต้องใช้ตอนทำนาย (pipeline + threshold + metrics) ไว้ในไฟล์เดียว `model.joblib` ให้ `predictss.py` โหลดไปใช้

ลำดับ: โหลดข้อมูล → เลือกฟีเจอร์ให้ตรงกับ API → แบ่ง train/test → เปรียบเทียบโมเดลด้วย cross-validation → เลือก threshold → วัดผลบน test → ตรวจ fairness → บันทึก artifact → ทดสอบแบบเดียวกับ API

In [1]:
import json
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=FutureWarning)

# Import shared feature code as `models.features` so the pickled pipeline resolves the same way inside the API.
BACKEND_DIR = Path.cwd().parent
sys.path.insert(0, str(BACKEND_DIR))
from models.features import (CATEGORICAL_FEATURES, ENGINEERED_FEATURES, FEATURES, NUMERIC_FEATURES,
                             add_engineered_features)

SEED = 42
DATA_PATH = Path("../../../data/Data/data.csv")
ARTIFACT_PATH = Path("model.joblib")
METRICS_PATH = Path("metrics.json")
MODEL_VERSION = "2.0.0"

## 1. โหลดข้อมูล

In [2]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print("missing values:", int(df.isnull().sum().sum()))
print(df["Attrition"].value_counts().to_frame("count").assign(ratio=lambda t: (t["count"] / len(df)).round(3)))

shape: (1470, 35)
missing values: 0
           count  ratio
Attrition              
No          1233  0.839
Yes          237  0.161


## 2. เลือกฟีเจอร์ให้ตรงกับ API

ใช้เฉพาะ 24 ฟิลด์ที่หน้าเว็บส่งมา ไม่ใช้เพศ สถานภาพสมรส แผนก หรือตำแหน่งงาน ตอนทำนายจึงไม่ต้องเติมค่าให้คอลัมน์ที่ API ไม่ได้ส่ง

In [3]:
app_source = (BACKEND_DIR / "app.py").read_text(encoding="utf-8")
api_fields = re.findall(r"^\s+(\w+)\s*:\s*\w+", re.search(r"class Pred_IP\(BaseModel\):(.*?)\n\S", app_source, re.S).group(1), re.M)
assert set(api_fields) == set(FEATURES), set(api_fields) ^ set(FEATURES)

X = df[FEATURES]
y = (df["Attrition"] == "Yes").astype(int).values
print(f"{len(FEATURES)} features ({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical) match Pred_IP in app.py")

24 features (23 numeric, 1 categorical) match Pred_IP in app.py


## 3. แบ่ง train / test (80 / 20, stratified)

test set ใช้วัดผลครั้งเดียวตอนท้าย การเลือกโมเดลและ threshold ทำบน train ด้วย cross-validation เท่านั้น

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
print(f"train: {len(X_train)} rows ({y_train.sum()} leave) | test: {len(X_test)} rows ({y_test.sum()} leave)")

train: 1176 rows (190 leave) | test: 294 rows (47 leave)


## 4. Pipeline

`add_engineered_features` (สัดส่วนอายุงาน, ระยะห่างจากการเลื่อนตำแหน่ง, รายได้ต่อระดับงาน ฯลฯ + log ของค่าที่เบ้) → `StandardScaler` / `OneHotEncoder` → classifier

ทุกขั้นอยู่ใน pipeline เดียว ตอนทำ cross-validation จึง fit scaler เฉพาะข้อมูลใน fold นั้น ไม่รั่วไปถึง fold ที่ใช้วัดผล

In [5]:
def build_pipeline(clf, engineered=True):
    numeric = NUMERIC_FEATURES + (ENGINEERED_FEATURES if engineered else [])
    steps = [("features", FunctionTransformer(add_engineered_features))] if engineered else []
    steps += [
        ("preprocess", ColumnTransformer([
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ])),
        ("clf", clf),
    ]
    return Pipeline(steps)

## 5. เปรียบเทียบโมเดล (Repeated Stratified 5-Fold × 3)

ข้อมูลไม่สมดุล (ลาออก 16%) จึงใช้ class weight แบบ balanced และเลือกโมเดลจาก PR-AUC (average precision) ซึ่งสะท้อนการหาคนที่จะลาออกได้ดีกว่า accuracy

In [6]:
cv_repeated = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
candidates = {
    "LogisticRegression": (LogisticRegression(class_weight="balanced", max_iter=5000),
                           {"clf__C": [0.01, 0.03, 0.1, 0.3, 1]}),
    "SVC (RBF)": (SVC(probability=True, class_weight="balanced", random_state=SEED),
                  {"clf__C": [0.3, 1, 3, 10], "clf__gamma": ["scale", 0.003, 0.01, 0.03]}),
    "HistGradientBoosting": (HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
                             {"clf__learning_rate": [0.03, 0.1], "clf__max_leaf_nodes": [7, 15],
                              "clf__l2_regularization": [0.0, 1.0]}),
}

searches, rows = {}, []
for name, (clf, grid) in candidates.items():
    for engineered in (False, True):
        label = f"{name} + engineered" if engineered else name
        gs = GridSearchCV(build_pipeline(clf, engineered), grid, cv=cv_repeated, n_jobs=-1,
                          scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc"}, refit="pr_auc")
        gs.fit(X_train, y_train)
        i = gs.best_index_
        searches[label] = gs
        rows.append({"model": label,
                     "cv_pr_auc": gs.cv_results_["mean_test_pr_auc"][i], "pr_auc_std": gs.cv_results_["std_test_pr_auc"][i],
                     "cv_roc_auc": gs.cv_results_["mean_test_roc_auc"][i], "roc_auc_std": gs.cv_results_["std_test_roc_auc"][i],
                     "best_params": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}})

cv_table = pd.DataFrame(rows)

LSTM แบบเดิม (โครงสร้างเดียวกับเวอร์ชันก่อน) วัดด้วย fold ชุดเดียวกันเพื่อเทียบ ต้องมี TensorFlow ในเครื่องที่รัน notebook เท่านั้น API ไม่ได้ใช้

In [7]:
try:
    import os
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    tf.get_logger().setLevel("ERROR")

    def lstm_fold(X_fit, y_fit, X_eval):
        prep = build_pipeline(None, engineered=False).steps[0][1].fit(X_fit)
        A = prep.transform(X_fit).reshape(-1, 1, len(FEATURES))
        B = prep.transform(X_eval).reshape(-1, 1, len(FEATURES))
        keras.utils.set_random_seed(SEED)
        net = keras.Sequential([layers.Input((1, len(FEATURES))), layers.LSTM(32), layers.Dropout(0.3),
                                layers.Dense(16, activation="relu"), layers.Dense(1, activation="sigmoid")])
        net.compile("adam", "binary_crossentropy", metrics=[keras.metrics.AUC(name="pr_auc", curve="PR")])
        net.fit(A, y_fit, validation_split=0.15, epochs=80, batch_size=32, class_weight={0: 1.0, 1: 5.0}, verbose=0,
                callbacks=[keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=10,
                                                         restore_best_weights=True)])
        return net.predict(B, verbose=0).ravel()

    pr, roc = [], []
    for fit_idx, eval_idx in cv_repeated.split(X_train, y_train):
        p = lstm_fold(X_train.iloc[fit_idx], y_train[fit_idx], X_train.iloc[eval_idx])
        pr.append(average_precision_score(y_train[eval_idx], p))
        roc.append(roc_auc_score(y_train[eval_idx], p))
    cv_table = pd.concat([cv_table, pd.DataFrame([{"model": "LSTM (previous architecture)", "cv_pr_auc": np.mean(pr),
                                                   "pr_auc_std": np.std(pr), "cv_roc_auc": np.mean(roc),
                                                   "roc_auc_std": np.std(roc), "best_params": {}}])], ignore_index=True)
except ImportError:
    print("TensorFlow not installed: skipping the LSTM comparison")

cv_table = cv_table.sort_values("cv_pr_auc", ascending=False).reset_index(drop=True)
cv_table.round(3)

,model,cv_pr_auc,pr_auc_std,cv_roc_auc,roc_auc_std,best_params
0,LogisticRegression + engineered,0.584,0.060,0.808,0.035,{'C': 0.3}
1,SVC (RBF) + engineered,0.579,0.049,0.812,0.030,"{'C': 10, 'gamma': 0.003}"
2,SVC (RBF),0.578,0.056,0.812,0.034,"{'C': 10, 'gamma': 0.003}"
3,HistGradientBoosting + engineered,0.572,0.056,0.792,0.028,"{'l2_regularization': 0.0, 'learning_rate': 0...."
4,LogisticRegression,0.550,0.067,0.800,0.037,{'C': 0.03}
5,HistGradientBoosting,0.538,0.052,0.781,0.030,"{'l2_regularization': 1.0, 'learning_rate': 0...."
6,LSTM (previous architecture),0.523,0.056,0.769,0.030,{}


## 6. เลือกโมเดลและ threshold

threshold มาจาก out-of-fold probability บน train (ค่าที่ทำให้ F1 สูงสุด) แล้วบันทึกไปกับโมเดล ตอนใช้งานจริงจึงใช้ค่าเดียวกับที่วิเคราะห์ไว้

In [8]:
best_name = cv_table.loc[cv_table["model"].isin(searches), "model"].iloc[0]
best_search = searches[best_name]
model = best_search.best_estimator_  # refit on the full training set by GridSearchCV
print("selected:", best_name, best_search.best_params_)

oof = cross_val_predict(model, X_train, y_train, method="predict_proba",
                        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED))[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, oof)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
threshold = float(thresholds[np.argmax(f1[:-1])])
print(f"threshold (max out-of-fold F1): {threshold:.3f}")

selected: LogisticRegression + engineered {'clf__C': 0.3}


threshold (max out-of-fold F1): 0.746


## 7. ผลบน test set

In [9]:
def scores(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {"roc_auc": roc_auc_score(y_true, proba), "pr_auc": average_precision_score(y_true, proba),
            "accuracy": accuracy_score(y_true, pred), "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred), "f1": f1_score(y_true, pred),
            "predicted_leave_rate": pred.mean()}

p_test = model.predict_proba(X_test)[:, 1]
test_metrics = scores(y_test, p_test, threshold)
pd.DataFrame({f"threshold {threshold:.3f}": test_metrics, "threshold 0.5": scores(y_test, p_test, 0.5)}).round(3)

,threshold 0.746,threshold 0.5
roc_auc,0.776,0.776
pr_auc,0.525,0.525
accuracy,0.854,0.745
precision,0.562,0.354
recall,0.383,0.723
f1,0.456,0.476
predicted_leave_rate,0.109,0.327


In [10]:
pred_test = (p_test >= threshold).astype(int)
cm = confusion_matrix(y_test, pred_test)
print(pd.DataFrame(cm, index=["actual Stay", "actual Leave"], columns=["pred Stay", "pred Leave"]))
print()
print(classification_report(y_test, pred_test, target_names=["Stay", "Leave"]))

              pred Stay  pred Leave
actual Stay         233          14
actual Leave         29          18

              precision    recall  f1-score   support

        Stay       0.89      0.94      0.92       247
       Leave       0.56      0.38      0.46        47

    accuracy                           0.85       294
   macro avg       0.73      0.66      0.69       294
weighted avg       0.84      0.85      0.84       294



## 8. เทียบกับโมเดลเดิมที่ deploy อยู่

โมเดลเดิม (`legacy/`) เทรนด้วย 44 คอลัมน์ แต่ API ส่งไปแค่ 24 ฟิลด์ คอลัมน์ที่เหลือถูกเติม 0 และใช้ threshold 0.5 วัดบน test set เดิมของมัน (แบ่ง 70/15/15 ด้วย seed เดียวกัน) ทั้งแบบที่ notebook เดิมรายงาน และแบบที่ API ใช้จริง

In [11]:
LEGACY_DIR = Path("legacy")
try:
    from tensorflow import keras
    legacy_model = keras.models.load_model(LEGACY_DIR / "model.h5")
    legacy_scaler = joblib.load(LEGACY_DIR / "scaler.pkl")
    legacy_columns = joblib.load(LEGACY_DIR / "model_columns.pkl")

    idx = np.arange(len(df))
    _, tmp_idx, _, y_tmp = train_test_split(idx, y, test_size=0.30, stratify=y, random_state=SEED)
    _, legacy_test_idx, _, y_legacy_test = train_test_split(tmp_idx, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

    def legacy_predict(rows):
        d = rows.drop(columns=["Attrition", "EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"], errors="ignore")
        num = list(legacy_scaler.feature_names_in_)
        for c in num:
            if c not in d:
                d[c] = 0
        d[num] = legacy_scaler.transform(d[num])
        d = pd.get_dummies(d, columns=d.select_dtypes(include=["object", "string"]).columns.tolist(), drop_first=True)
        d = d.reindex(columns=legacy_columns, fill_value=0)
        return legacy_model.predict(d.values.astype(np.float32).reshape(-1, 1, len(legacy_columns)), verbose=0).ravel()

    legacy_rows = df.iloc[legacy_test_idx]
    legacy_table = pd.DataFrame({
        "old notebook (44 columns, thr 0.803)": scores(y_legacy_test, legacy_predict(legacy_rows), 0.803),
        "old API in production (24 fields + zeros, thr 0.5)": scores(y_legacy_test, legacy_predict(legacy_rows[FEATURES]), 0.5),
        "new pipeline on this test set": scores(y_legacy_test, model.predict_proba(X.iloc[legacy_test_idx])[:, 1], threshold),
    }).round(3)
    print("note: the new pipeline trained on part of the old test rows, so its column here is optimistic;"
          " section 7 is its fair score")
    display(legacy_table)
except (ImportError, OSError) as e:
    print("legacy comparison skipped:", e)

note: the new pipeline trained on part of the old test rows, so its column here is optimistic; section 7 is its fair score


,"old notebook (44 columns, thr 0.803)","old API in production (24 fields + zeros, thr 0.5)",new pipeline on this test set
roc_auc,0.806,0.798,0.780
pr_auc,0.626,0.608,0.559
accuracy,0.864,0.629,0.873
precision,0.594,0.279,0.667
recall,0.528,0.806,0.444
f1,0.559,0.414,0.533
predicted_leave_rate,0.145,0.471,0.109


## 9. Fairness check บน test set

โมเดลไม่ได้ใช้เพศ สถานภาพสมรส หรือแผนก แต่ยังใช้อายุ (เพราะ API รับค่าอายุ) จึงตรวจว่าแต่ละกลุ่มถูกทำนายว่าเสี่ยงต่างกันแค่ไหน และ recall ต่างกันแค่ไหน

In [12]:
audit = df.loc[X_test.index, ["Gender", "MaritalStatus", "Age"]].assign(
    AgeGroup=lambda d: pd.cut(d["Age"], [0, 29, 44, 100], labels=["< 30", "30-44", "45+"]),
    actual=y_test, predicted=pred_test)

def group_report(col):
    g = audit.groupby(col, observed=True)
    return pd.DataFrame({
        "n": g.size(),
        "actual_leave_rate": g["actual"].mean(),
        "predicted_leave_rate": g["predicted"].mean(),
        "recall": g.apply(lambda t: recall_score(t["actual"], t["predicted"]) if t["actual"].sum() else np.nan),
    }).round(3)

fairness = {col: group_report(col) for col in ["Gender", "MaritalStatus", "AgeGroup"]}
for col, table in fairness.items():
    print(f"--- {col} ---")
    print(table, end="\n\n")

--- Gender ---
          n  actual_leave_rate  predicted_leave_rate  recall
Gender                                                      
Female  116              0.138                 0.121   0.375
Male    178              0.174                 0.101   0.387

--- MaritalStatus ---
                 n  actual_leave_rate  predicted_leave_rate  recall
MaritalStatus                                                      
Divorced        64              0.062                 0.047   0.250
Married        133              0.135                 0.105   0.389
Single          97              0.258                 0.155   0.400

--- AgeGroup ---
            n  actual_leave_rate  predicted_leave_rate  recall
AgeGroup                                                      
< 30       68              0.250                 0.221   0.471
30-44     173              0.121                 0.087   0.381
45+        53              0.170                 0.038   0.222



## 10. ฟีเจอร์ที่มีผลต่อโมเดล (permutation importance บน test set)

In [13]:
imp = permutation_importance(model, X_test, y_test, scoring="average_precision", n_repeats=30, random_state=SEED, n_jobs=-1)
importance = (pd.DataFrame({"feature": FEATURES, "importance": imp.importances_mean, "std": imp.importances_std})
              .sort_values("importance", ascending=False).reset_index(drop=True))
importance.head(10).round(4)

,feature,importance,std
0,TotalWorkingYears,0.1943,0.0484
1,OverTime,0.1880,0.0262
2,YearsAtCompany,0.1320,0.0340
3,NumCompaniesWorked,0.0718,0.0216
4,JobSatisfaction,0.0584,0.0200
5,EnvironmentSatisfaction,0.0522,0.0272
6,JobInvolvement,0.0313,0.0188
7,StockOptionLevel,0.0296,0.0183
8,DistanceFromHome,0.0229,0.0174
9,JobLevel,0.0210,0.0211


## 11. บันทึก artifact

`model.joblib` เก็บ pipeline, threshold, รายชื่อฟีเจอร์ และคะแนน ส่วน `metrics.json` เก็บคะแนนไว้ให้อ่านได้โดยไม่ต้องโหลดโมเดล

In [14]:
metrics = {
    "model_version": MODEL_VERSION,
    "model": best_name,
    "params": {k.replace("clf__", ""): v for k, v in best_search.best_params_.items()},
    "threshold": round(threshold, 4),
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "features": FEATURES,
    "data": {"rows": len(df), "train_rows": len(X_train), "test_rows": len(X_test), "leave_rate": round(float(y.mean()), 4)},
    "cv": cv_table.drop(columns="best_params").round(4).to_dict(orient="records"),
    "test": {k: round(float(v), 4) for k, v in test_metrics.items()},
    "confusion_matrix": {"tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1])},
    "top_features": importance.head(10)[["feature", "importance"]].round(4).to_dict(orient="records"),
}
joblib.dump({"pipeline": model, "threshold": threshold, "features": FEATURES, "metrics": metrics}, ARTIFACT_PATH, compress=3)
METRICS_PATH.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"saved {ARTIFACT_PATH} ({ARTIFACT_PATH.stat().st_size / 1024:.1f} KB) and {METRICS_PATH}")

saved model.joblib (3.7 KB) and metrics.json


## 12. ทดสอบแบบเดียวกับที่ API เรียก

In [15]:
from models.predictss import predict_model, predict_proba

for i in [0, 1]:  # row 0 left the company, row 1 stayed
    payload = {k: (v.item() if hasattr(v, "item") else v) for k, v in df.loc[i, FEATURES].items()}
    print(f"row {i}: actual={df.loc[i, 'Attrition']}  probability={predict_proba(payload):.3f}  prediction={predict_model(payload)}")

row 0: actual=Yes  probability=0.777  prediction=1
row 1: actual=No  probability=0.128  prediction=0
